# ML-02 — Research Question and Provisional Lane

**Project:** Content Refresh / Opportunity Scoring

This notebook explores how search-performance data can help a content team decide which pages to review first. It establishes a simple, rule-based baseline; it does **not** train or validate a machine-learning model.

[Open this notebook in Google Colab](https://colab.research.google.com/github/dev-anshu11/flyrank-ml-internship/blob/main/work/notebooks/w01_research_question.ipynb)


## 1. My lane and why I chose it

**Selected lane: Lane 2 — Refresh / Content Opportunity Scoring**

I chose this lane because content teams need a practical way to decide which pages deserve review first. I want to explore whether search-performance signals can help create a useful review shortlist. I will begin with data exploration and a transparent rule-based baseline. Later, I can investigate whether a machine-learning approach improves the shortlist. The lane is provisional, so I may refine the question as I learn more about the data.


## 2. Research question: decision, action, and cost of a wrong call

### Research question

**How can we use available search-performance data to identify and prioritize web pages that may deserve a content review?**

- **Decision:** Which pages should the content team review first?
- **Who acts:** A content manager or content editor reviews the shortlist.
- **Possible action:** The team checks whether a page needs updated information, clearer writing, additional coverage, or continued monitoring. The analysis does not automatically tell the team to edit a page.
- **Cost of a false positive:** The team may spend time reviewing a page that does not need attention.
- **Cost of a false negative:** A page that may deserve attention could be missed.
- **Why data may help:** Impressions, clicks, sessions, CTR, engagement measures, and content age may provide useful signals for deciding where to investigate.

Machine learning is a possible later step. First, I need to understand the available fields, data quality, and whether a simple baseline provides a sensible starting point.


## 3. Quick look at the starter data

The following cell loads the repository's starter CSV and reports basic dataset facts. These are descriptive statistics, not model results.


In [1]:
import pandas as pd
import numpy as np

# Load the starter dataset directly from the project repository
data_url = (
    "https://raw.githubusercontent.com/"
    "dev-anshu11/flyrank-ml-internship/main/"
    "data/raw/content_refresh_anonymized.csv"
)

df = pd.read_csv(data_url)

print("Dataset loaded successfully")
print("----------------------------")
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")
print(f"Unique clients: {df['client_id'].nunique()}")

print("\nTrend direction:")
print(df["trend_direction"].value_counts(dropna=False))

down_count = (df["trend_direction"] == "down").sum()
down_pct = down_count / len(df) * 100

print(f"\nDownward-trending records: {down_count:,} ({down_pct:.2f}%)")


Dataset loaded successfully
----------------------------
Rows: 30,000
Columns: 44
Unique clients: 32

Trend direction:
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64

Downward-trending records: 16,262 (54.21%)


### Initial observations

The starter dataset contains **30,000 rows**, **44 columns**, and **32 anonymized clients**. In the initial exploration, **16,262 records (54.21%)** are labelled with a downward trend.

A downward trend is a signal for investigation; it does not prove that a page needs refreshing.


In [2]:
# Check the zero average-position values before using position in the analysis

zero_position = df[df["avg_position"] == 0].copy()

print("Rows with avg_position = 0:", len(zero_position))
print("Median impressions over 90 days:", zero_position["impressions_90d"].median())
print("Maximum impressions over 90 days:", zero_position["impressions_90d"].max())
print("\nPosition-tier labels among zero-position rows:")
print(zero_position["position_tier"].value_counts(dropna=False))


Rows with avg_position = 0: 1205
Median impressions over 90 days: 1.0
Maximum impressions over 90 days: 36

Position-tier labels among zero-position rows:
position_tier
top_3    1205
Name: count, dtype: int64


### Data-quality decision

There are **1,205 records** where `avg_position` is 0, even though the position-tier field labels these rows as `top_3`. A zero is not a usable search-position value, so I treat it as unknown for the ranking exercise rather than interpreting it as a real position.

This is a data-cleaning choice. It does not mean those pages have no value.


In [3]:
# Create a cleaned position field: zero becomes missing/unknown

df["avg_position_clean"] = df["avg_position"].replace(0, np.nan)

print("Original zero-position rows:", (df["avg_position"] == 0).sum())
print("Unknown positions after cleaning:", df["avg_position_clean"].isna().sum())


Original zero-position rows: 1205
Unknown positions after cleaning: 1205


## 4. Build a transparent rule-based baseline

For this first baseline, I use the following rules:

1. Keep records labelled with a downward trend.
2. Calculate the decrease in impressions as previous 30-day impressions minus latest 30-day impressions.
3. Exclude records where average position is unknown.
4. Rank the remaining pages by impression loss.
5. For a balanced review shortlist, keep no more than two pages per client.
6. Apply a **1,000-impression minimum loss** and select up to 50 pages.

The 1,000-impression cutoff and two-pages-per-client limit are choices for this exploratory baseline. They are not validated thresholds, and this ranking is not a prediction.


In [4]:
# Step 1: Keep downward-trending pages and calculate impression loss

down = df[df["trend_direction"] == "down"].copy()

down["impressions_lost"] = (
    down["impressions_prev_30d"] - down["impressions_last_30d"]
)

# Step 2: Exclude pages with unknown average position
eligible = down[down["avg_position_clean"].notna()].copy()

# Step 3: Rank eligible pages by impression loss
ranked = eligible.sort_values(
    "impressions_lost",
    ascending=False
).copy()

print("Downward-trending pages:", len(down))
print("Eligible pages after excluding unknown position:", len(ranked))
print("Excluded downward-trending pages:", len(down) - len(ranked))


Downward-trending pages: 16262
Eligible pages after excluding unknown position: 16254
Excluded downward-trending pages: 8


In [5]:
# Inspect the excluded downward-trending pages.
# Check their actual impression losses before making any claim about their size.

excluded = down[down["avg_position_clean"].isna()].copy()

print("Excluded pages:", len(excluded))
if len(excluded) > 0:
    print(
        "Maximum impression loss among excluded pages:",
        (excluded["impressions_prev_30d"] - excluded["impressions_last_30d"]).max()
    )

    excluded["impressions_lost"] = (
        excluded["impressions_prev_30d"] - excluded["impressions_last_30d"]
    )

    display(
        excluded[
            ["content_id", "impressions_lost", "avg_position"]
        ].sort_values("impressions_lost", ascending=False)
    )


Excluded pages: 8
Maximum impression loss among excluded pages: 6


,content_id,impressions_lost,avg_position
26684,content_6edec030867e,6,0.0
7006,content_d964111653a1,3,0.0
18985,content_d19d9bb94617,2,0.0
5770,content_28b4223f4e5f,1,0.0
3879,content_34b14c00f80c,1,0.0
24460,content_31c8f34527e2,1,0.0
26818,content_6e6f5c5f7a06,1,0.0
27271,content_7bc32bc1df59,1,0.0


In [6]:
# Compare an unrestricted top-50 list with the client-balanced approach

unrestricted_top50 = ranked.head(50)

print("Clients represented in unrestricted top 50:",
      unrestricted_top50["client_id"].nunique())
print("\nPages per client in unrestricted top 50:")
print(unrestricted_top50["client_id"].value_counts())


Clients represented in unrestricted top 50: 7

Pages per client in unrestricted top 50:
client_id
client_7f2253d7e2    13
client_19581e27de    10
client_6208ef0f77    10
client_4e07408562     9
client_f369cb89fc     3
client_349c41201b     3
client_b4944c6ff0     2
Name: count, dtype: int64


In [7]:
# Create a client-balanced shortlist using the chosen rules

ranked["client_rank"] = ranked.groupby("client_id")["impressions_lost"].rank(
    ascending=False,
    method="first"
)

client_balanced = ranked[ranked["client_rank"] <= 2].copy()

final_review_list = client_balanced[
    client_balanced["impressions_lost"] >= 1000
].sort_values(
    "impressions_lost",
    ascending=False
).head(50).copy()

print("Final selected pages:", len(final_review_list))
print("Clients represented:", final_review_list["client_id"].nunique())
if len(final_review_list) > 0:
    print("Smallest impression loss:",
          final_review_list["impressions_lost"].min())


Final selected pages: 41
Clients represented: 21
Smallest impression loss: 1151


In [8]:
# Final results summary

print("FINAL CONTENT REVIEW BASELINE")
print("------------------------------")
print("Total dataset rows:", len(df))
print("Downward-trending pages:", len(down))
print("Eligible pages for ranking:", len(ranked))
print("Final selected pages:", len(final_review_list))
print("Clients represented:", final_review_list["client_id"].nunique())

if len(final_review_list) > 0:
    print("Smallest impression loss:",
          final_review_list["impressions_lost"].min())


FINAL CONTENT REVIEW BASELINE
------------------------------
Total dataset rows: 30000
Downward-trending pages: 16262
Eligible pages for ranking: 16254
Final selected pages: 41
Clients represented: 21
Smallest impression loss: 1151


In [11]:
# Export the final shortlist for manual review

export_columns = [
    "content_id",
    "client_id",
    "impressions_prev_30d",
    "impressions_last_30d",
    "impressions_lost",
    "avg_position_clean",
    "client_rank"
]

final_review_list[export_columns].to_csv(
    "final_content_review_shortlist.csv",
    index=False
)

print("Exported: final_content_review_shortlist.csv")
print("Rows exported:", len(final_review_list))


Exported: final_content_review_shortlist.csv
Rows exported: 41


## 5. What I can and cannot claim

### What this analysis supports

- The starter dataset contains 30,000 rows, 44 columns, and 32 anonymized clients.
- 16,262 records are labelled as downward-trending in the provided data.
- Some average-position values are zero and need careful treatment before using position as a ranking signal.
- A transparent set of rules can produce a shortlist for human review.
- The shortlist reflects the chosen filters and ranking rules.

### What this analysis does not prove

- It does not prove that every downward-trending page needs refreshing.
- It does not prove that a content update will increase impressions, clicks, or search ranking.
- It does not establish that impression loss is caused by any one feature.
- It does not predict Google's ranking decisions.
- It is not yet a trained or validated machine-learning model.

### Important limitations

The data is observational, and the ranking rules are provisional. The 1,000-impression cutoff and two-pages-per-client limit were selected for this exercise; they have not been validated against editor decisions or later page performance. A human should review the shortlist before taking action.


## 6. Self-check and next step

1. **Is the question connected to a real decision?** Yes. It helps a content team decide which pages to review first.
2. **Who takes action?** A content manager or editor reviews the shortlist and decides what, if anything, to change.
3. **Have I considered the cost of a wrong recommendation?** Yes. False positives waste review time; false negatives may leave a useful opportunity unnoticed.
4. **Have I used actual dataset information?** Yes. The notebook reports the row count, column count, client count, trend counts, and zero-position records from the starter CSV.
5. **Have I avoided unsupported claims?** Yes. This is a descriptive, rule-based baseline, not proof of causation or a predictive model.
6. **What will I investigate next?** I will examine which available features are useful, check data quality and possible leakage, and define a fair way to evaluate a future baseline or machine-learning model.


## 7. Reflection: which mistake is more costly?

I currently think ranking a page too low may be the more costly mistake because a page that deserves attention could be missed. However, ranking a page too high also has a cost: the team may spend time reviewing a page unnecessarily. The relative cost should be discussed with the content team rather than assumed from this dataset alone.

The 16,262 downward-trending records show that the label is common in this dataset. They do not tell me how many pages were actually missed by a review team, so I cannot quantify the cost of either mistake yet.
